In [1]:
import pandas as pd
import re
import os
import numpy as np
import requests

In [2]:
os.getcwd()
#Num format
pd.options.display.float_format = '{:,.2f}'.format

path="scraping_output/final"

In [3]:
def clean_data(df):
    # Lowercase name and size
    df['name'] = df['name'].str.lower()
    df['size'] = df['size'].str.lower()

    #Keep apartments and houses
    df = df[df['type'].isin(['Apartment', 'House'])]

    def parse_number(val):
        """Parse number string handling dot/comma as thousand or decimal separator."""
        match = re.match(r'^[\d.,]+', str(val).strip())
        if not match:
            return np.nan
        s = match.group()

        has_dot = '.' in s
        has_comma = ',' in s

        if has_dot and has_comma:
            if s.rfind('.') > s.rfind(','):
                # e.g. 1,047.89 → comma=miles, dot=decimal
                s = s.replace(',', '')
            else:
                # e.g. 1.047,89 → dot=miles, comma=decimal
                s = s.replace('.', '').replace(',', '.')
        elif has_dot:
            # dot=thousands if ALL groups after dots have exactly 3 digits
            parts = s.split('.')
            if all(len(p) == 3 for p in parts[1:]):
                s = s.replace('.', '')
        elif has_comma:
            parts = s.split(',')
            if all(len(p) == 3 for p in parts[1:]):
                s = s.replace(',', '')
            else:
                s = s.replace(',', '.')

        return pd.to_numeric(s, errors='coerce')

    # Extract currency prefix (e.g. "USD") from price string and update currency column
    def extract_currency(row):
        val = row['price']
        if pd.isna(val):
            return row['currency'], val
        s = str(val).strip()
        m = re.match(r'^([A-Z]{2,4})\s*([\d.,].*)$', s)
        if m:
            detected = m.group(1)
            price_str = m.group(2)
            # Keep existing currency if already set, otherwise use detected one
            currency = row['currency'] if pd.notna(row['currency']) else detected
            return currency, price_str
        return row['currency'], val

    extracted = df[['price', 'currency']].apply(extract_currency, axis=1, result_type='expand')
    df['currency'] = extracted[0]
    df['price'] = extracted[1]

    # Convert price to numeric: remove "$" then parse
    def parse_price(val):
        if pd.isna(val):
            return np.nan
        return parse_number(str(val).replace('$', ''))

    # Convert size to numeric: remove "m²"/"m2" then parse
    def parse_size(val):
        if pd.isna(val):
            return np.nan
        return parse_number(str(val).replace('m²', '').replace('m2', ''))

    df['price'] = df['price'].apply(parse_price)
    df['size'] = df['size'].apply(parse_size)

    # Drop rows where price or size is NaN
    df = df.dropna(subset=['price', 'size'])

    return df

In [4]:
##Load raw data

raw=pd.read_csv("scraping_output/raw/total_scraping.csv", low_memory=False)
print("Total rows", len(raw))
raw.head()

Total rows 63277


,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,region,locality,consultation_date,source,country,operation
0,Departamento en Venta en La Carolina,$ 140,NaN,Apartment,70 m²,1.00,2.00,-0.19,-78.49,"Eloy Alfaro Y de la Republica, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
1,Departamento en Venta en San Isidro del Inca,$ 69.900,NaN,Apartment,69 m²,3.00,2.00,-0.14,-78.46,"Calle N50G, Quito, ECU",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
2,Departamento en Venta en Centro De Quito,$ 68.000,NaN,Apartment,106 m²,3.00,2.00,-0.21,-78.50,"Asunción & Venezuela, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
3,Oficina en Venta en La Mariscal,$ 140.000,NaN,Accommodation,166 m²,7.00,2.00,-0.20,-78.49,Av. Francisco de Orellana & Avenida 10 de Agos...,Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
4,Departamento en Venta en Nayón,$ 110.000,NaN,Apartment,115 m²,2.00,3.00,-0.16,-78.44,"Jaime Roldos Aguilera & 19 de Diciembre, Quito...",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell


In [5]:
raw["currency"].unique()

array([nan, 'BRL'], dtype=object)

In [6]:
print("Data before cleaning")
print(raw.groupby(['country', 'operation']).size().unstack(fill_value=0).to_string())

Data before cleaning
operation    rent  sell
country                
Argentina    3000  3000
Brazil       1200  1200
Chile        3000  3000
Colombia     3000  3000
Costa-rica      0  3001
Ecuador      3000  3000
El-salvador     0  2982
Guatemala       0  3000
Honduras        0  2985
Mexico       5165  7768
Nicaragua       0  2974
Panama          0  3002
Peru         3000  3000


In [7]:
## Clean data
clean_scrap=clean_data(raw.copy())
print("Total rows:", len(clean_scrap))
clean_scrap

Total rows: 36310


,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,region,locality,consultation_date,source,country,operation
0,departamento en venta en la carolina,140.00,NaN,Apartment,70.00,1.00,2.00,-0.19,-78.49,"Eloy Alfaro Y de la Republica, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
1,departamento en venta en san isidro del inca,"69,900.00",NaN,Apartment,69.00,3.00,2.00,-0.14,-78.46,"Calle N50G, Quito, ECU",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
2,departamento en venta en centro de quito,"68,000.00",NaN,Apartment,106.00,3.00,2.00,-0.21,-78.50,"Asunción & Venezuela, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
4,departamento en venta en nayón,"110,000.00",NaN,Apartment,115.00,2.00,3.00,-0.16,-78.44,"Jaime Roldos Aguilera & 19 de Diciembre, Quito...",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell
5,casa en venta en garzota,"350,000.00",NaN,House,622.00,15.00,9.00,-2.15,-79.89,"Ciudadela Ietel, IETEL, Guayaquil, Ecuador",Guayas,Guayaquil,2026-03-19 18:26:04,Properati,Ecuador,sell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63262,2d1b arriendo departamento cercano a todo,"365,000.00",NaN,Apartment,51.00,2.00,1.00,NaN,NaN,NaN,NaN,Santiago,2026-03-20 03:12:43,Yapo,Chile,rent
63263,"cómodo, con excelente conectividad, pasos del...","380,000.00",NaN,Apartment,40.00,1.00,1.00,NaN,NaN,NaN,NaN,Santiago,2026-03-20 03:12:43,Yapo,Chile,rent
63264,departamento comandante malbec y av. la dehesa,60.00,UF,Apartment,155.00,2.00,3.00,NaN,NaN,NaN,NaN,Lo Barnechea,2026-03-20 03:12:43,Yapo,Chile,rent
63266,"departamento en arriendo, recreo alto","520,000.00",NaN,Apartment,85.00,3.00,2.00,NaN,NaN,NaN,NaN,Valparaíso,2026-03-20 03:12:43,Yapo,Chile,rent


In [8]:
chile=raw[raw["country"]=="Chile"]
chile

,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,region,locality,consultation_date,source,country,operation
57277,"Departamento de 3 dormitorios, 1 baño, logia y...",$60.000.000,NaN,Apartment,56 m2,3.00,1.00,NaN,NaN,NaN,NaN,Maipú,2026-03-20 03:12:43,Yapo,Chile,sell
57278,SE VENDE DEPARTAMENTO EN SECTOR NORTE DE ANTOF...,"UF3.100,00",NaN,Apartment,42 m2,2.00,1.00,NaN,NaN,NaN,NaN,Antofagasta,2026-03-20 03:12:43,Yapo,Chile,sell
57279,"Deptos. 1D y 1 B, sin uso, a pasos de Metro P....","UF1.614,00Oportunidad",NaN,Apartment,29 m2,1.00,1.00,NaN,NaN,NaN,NaN,Independencia,2026-03-20 03:12:43,Yapo,Chile,sell
57280,Departamento BELLAVISTA / LORETO,"UF2.600,00",NaN,Apartment,38 m2,1.00,1.00,NaN,NaN,NaN,NaN,Recoleta,2026-03-20 03:12:43,Yapo,Chile,sell
57281,Hermoso Departamento en Venta sector Enjoy 3 d...,"UF4.150,00-5%",NaN,Apartment,80 m2,3.00,2.00,NaN,NaN,NaN,NaN,Coquimbo,2026-03-20 03:12:43,Yapo,Chile,sell
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63272,"DEPARTAMENTO ESTUDIO , AV. ALEMANIA.TEMUCO (14...","UF10,00",NaN,Apartment,NaN,1.00,1.00,NaN,NaN,NaN,NaN,Temuco,2026-03-20 03:12:43,Yapo,Chile,rent
63273,C. San Cristóbal 122 (1 Est /1 Bod) [2911-4],"$480,000",NaN,Apartment,NaN,2.00,2.00,NaN,NaN,NaN,NaN,Independencia,2026-03-20 03:12:43,Yapo,Chile,rent
63274,ARRIENDO MARZO/DICIEMBRE (143366),$750.000-6%,NaN,Apartment,NaN,3.00,3.00,NaN,NaN,NaN,NaN,Viña del Mar,2026-03-20 03:12:43,Yapo,Chile,rent
63275,Carnot [2839-2],"$331,000",NaN,Apartment,NaN,1.00,1.00,NaN,NaN,NaN,NaN,San Miguel,2026-03-20 03:12:43,Yapo,Chile,rent


In [9]:
currency_map = {
    'Argentina': 'ARS', 'Brazil': 'BRL', 'Chile': 'CLP',
    'Colombia': 'COP', 'Costa-rica': 'CRC', 'Ecuador': 'USD',
    'El-salvador': 'USD', 'Guatemala': 'GTQ', 'Honduras': 'HNL',
    'Mexico': 'MXN', 'Nicaragua': 'NIO', 'Panama': 'USD', 'Peru': 'PEN'
}

In [10]:
# Get exchange rates
API_KEY="cf53b6293b066860dd66b072"
url = f"https://v6.exchangerate-api.com/v6/{API_KEY}/latest/USD"
r = requests.get(url).json()

if r.get('result') == 'success':
    rates = r['conversion_rates']
    last_update = pd.to_datetime(r['time_last_update_utc']).strftime('%Y-%m-%d')

    exchange_table = pd.DataFrame([
        {
            'country': country,
            'rate': 1.0 if code == 'USD' else (1 / rates[code] if code in rates else None),
            'last_update': last_update
        }
        for country, code in currency_map.items()
    ])

    print(exchange_table.to_string(index=False))
else:
    print("⚠️ Error al obtener tasas:", r)

    country  rate last_update
  Argentina  0.00  2026-04-17
     Brazil  0.20  2026-04-17
      Chile  0.00  2026-04-17
   Colombia  0.00  2026-04-17
 Costa-rica  0.00  2026-04-17
    Ecuador  1.00  2026-04-17
El-salvador  1.00  2026-04-17
  Guatemala  0.13  2026-04-17
   Honduras  0.04  2026-04-17
     Mexico  0.06  2026-04-17
  Nicaragua  0.03  2026-04-17
     Panama  1.00  2026-04-17
       Peru  0.29  2026-04-17


In [11]:
clean_scrap = clean_scrap.merge(
    exchange_table[['country', 'rate', 'last_update']],
    on='country',
    how='left'
)

# Create price_usd:
# Create price_usd:
# - currency == 'USD' → direct price
# - currency == 'UF'  → price * uf_value (CLP) * rate_to_usd del país (CLP→USD)
# - NaN o moneda local → price * rate_to_usd
clean_scrap['price_usd'] = clean_scrap.apply(
    lambda row: row['price'] if row['currency'] == 'USD'
    #UF from Banco Central de Chile
    #https://tinyurl.com/yz5ffy3n
                else row['price'] *39841.72* row['rate'] if row['currency'] == 'UF'
                else row['price'] * row['rate'],
    axis=1
)

#Price per square meter
clean_scrap['price_per_sq_meter'] = clean_scrap['price_usd'] / clean_scrap['size']


clean_scrap

,name,price,currency,type,size,bedrooms,bathrooms,latitude,longitude,street,region,locality,consultation_date,source,country,operation,rate,last_update,price_usd,price_per_sq_meter
0,departamento en venta en la carolina,140.00,NaN,Apartment,70.00,1.00,2.00,-0.19,-78.49,"Eloy Alfaro Y de la Republica, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,1.00,2026-04-17,140.00,2.00
1,departamento en venta en san isidro del inca,"69,900.00",NaN,Apartment,69.00,3.00,2.00,-0.14,-78.46,"Calle N50G, Quito, ECU",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,1.00,2026-04-17,"69,900.00","1,013.04"
2,departamento en venta en centro de quito,"68,000.00",NaN,Apartment,106.00,3.00,2.00,-0.21,-78.50,"Asunción & Venezuela, Quito, Ecuador",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,1.00,2026-04-17,"68,000.00",641.51
3,departamento en venta en nayón,"110,000.00",NaN,Apartment,115.00,2.00,3.00,-0.16,-78.44,"Jaime Roldos Aguilera & 19 de Diciembre, Quito...",Pichincha,Quito,2026-03-19 18:26:04,Properati,Ecuador,sell,1.00,2026-04-17,"110,000.00",956.52
4,casa en venta en garzota,"350,000.00",NaN,House,622.00,15.00,9.00,-2.15,-79.89,"Ciudadela Ietel, IETEL, Guayaquil, Ecuador",Guayas,Guayaquil,2026-03-19 18:26:04,Properati,Ecuador,sell,1.00,2026-04-17,"350,000.00",562.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36305,2d1b arriendo departamento cercano a todo,"365,000.00",NaN,Apartment,51.00,2.00,1.00,NaN,NaN,NaN,NaN,Santiago,2026-03-20 03:12:43,Yapo,Chile,rent,0.00,2026-04-17,412.31,8.08
36306,"cómodo, con excelente conectividad, pasos del...","380,000.00",NaN,Apartment,40.00,1.00,1.00,NaN,NaN,NaN,NaN,Santiago,2026-03-20 03:12:43,Yapo,Chile,rent,0.00,2026-04-17,429.25,10.73
36307,departamento comandante malbec y av. la dehesa,60.00,UF,Apartment,155.00,2.00,3.00,NaN,NaN,NaN,NaN,Lo Barnechea,2026-03-20 03:12:43,Yapo,Chile,rent,0.00,2026-04-17,"2,700.35",17.42
36308,"departamento en arriendo, recreo alto","520,000.00",NaN,Apartment,85.00,3.00,2.00,NaN,NaN,NaN,NaN,Valparaíso,2026-03-20 03:12:43,Yapo,Chile,rent,0.00,2026-04-17,587.40,6.91


In [12]:
clean_scrap["currency"].unique()

array([nan, 'USD', 'HNL', 'BRL', 'UF'], dtype=object)

In [13]:
print("\nData after cleaning")
print(clean_scrap.groupby(['country', 'operation']).size().unstack(fill_value=0).to_string())


Data after cleaning
operation    rent  sell
country                
Argentina     761  1043
Brazil       1200  1200
Chile        1582  2840
Colombia     1356  1368
Costa-rica      0  2673
Ecuador      1383  1877
El-salvador     0  2978
Guatemala       0  2092
Honduras        0  2980
Mexico        352  2895
Nicaragua       0  2769
Panama          0  2999
Peru          503  1459


In [14]:
## Save clean_data
clean_scrap.to_csv(path + "/clean_scrap.csv",  index=False, encoding="utf-8-sig")